# **SN_AI – Predicting Serie A Match Results Using RNNs**

---

## Introduction / Context
The **SN_AI** project aims to predict Serie A match results from the 2015/2016 season to the ongoing 2025/2026 season.  
We will use **Recurrent Neural Networks (RNNs)** to capture temporal dependencies in historical team data and generate probabilistic match outcome predictions.

**Note:** Match results depend on more than historical results. Real-life factors such as player fitness, absences, strategies, and transfer market dynamics cannot currently be modeled due to insufficient data.

---

## Dataset and Split
- **Dataset:** each match is represented as a JSON object containing date, teams, score, and match number.  

```json
{
  "MatchNumber": 1,
  "RoundNumber": 1,
  "DateUtc": "2017-08-19 16:00:00Z",
  "Location": "Juventus Stadium",
  "HomeTeam": "Juventus",
  "AwayTeam": "Cagliari",
  "Group": null,
  "HomeTeamScore": 3,
  "AwayTeamScore": 0
}
```

**The dataset is divided in this way:**
- 70% for training
- 15% for validation
- 15% for testing

## Features
Main features used for the RNN:

- **Teams:** `home_team_name`, `away_team_name`  
- **Recent performance (last 5 matches):**  
  `home_last5_points`, `away_last5_points`  
  `home_last5_goals`, `away_last5_goals`  
- **Season averages:**  
  `home_avg_goals_scored`, `away_avg_goals_scored`  
  `home_avg_goals_conceded`, `away_avg_goals_conceded`  
- **Elo ratings:**  
  `elo_home_team`, `elo_away_team`, `elo_diff`  
- **Recent performance differences:**  
  `goal_diff_last5`, `points_diff_last5`  
- **Match history last 2 seasons:**  
  `last2yrs_match_history`  

---

## RNN Output
The network should output a Python object:

```python
result_predicted = {
    "home_win_prob": ...,
    "away_win_prob": ...,
    "draw_prob": ...,
    "home_goal_prediction": ...,
    "away_goal_prediction": ...,
    "over_2_5_prob": ...,
    "under_2_5_prob": ...
}
```

## Architecture and Method
- Network type: **RNN** (to capture temporal dependencies)  
- Activation function: **ReLU**  
- Loss function: **cross-entropy** (for result probabilities) and **MSE** (for goals)  
- Backpropagation: **BPTT (Backpropagation Through Time)**  
- Optimizer: **Adam**  
- Regularization: **dropout and L2 weight decay**  
- Validation: train/validation/test as indicated

---

## Evaluation
- Metrics:
  - Accuracy / F1-score / Precision / Recall (for win/draw/loss probabilities)  
  - MAE / MSE (for predicted goals)  
  - Confusion matrix  
- Feature importance analysis to understand which features most influence predictions

---

In [1]:
from pathlib import Path

def read_file(file_path: Path) -> str:
    """Read and return the content of a file."""
    print(f"Reading file: {file_path}")
    return file_path.read_text(encoding="utf-8")


def read_folder_files(folder_path: str) -> dict:
    """Read all files in a folder and return a dict {filename: content}."""
    folder = Path(folder_path)

    return {
        file.name: read_file(file)
        for file in folder.iterdir()
        if file.is_file()
    }

############################################################################################################################################################

train_path = "./json_files/train"
eval_path = "./json_files/eval"
testing_path = "./json_files/testing"

train_file_contents = read_folder_files(train_path)
eval_file_contents = read_folder_files(eval_path)
testing_file_contents = read_folder_files(testing_path)

Reading file: json_files/train/SerieA_9798.json
Reading file: json_files/train/SerieA_9900.json
Reading file: json_files/train/SerieA_9596.json
Reading file: json_files/train/SerieA_9899.json
Reading file: json_files/train/SerieA_9495.json
Reading file: json_files/train/SerieA_0607.json
Reading file: json_files/train/SerieA_9697.json
Reading file: json_files/train/SerieA_0708.json
Reading file: json_files/train/SerieA_0102.json
Reading file: json_files/train/SerieA_0304.json
Reading file: json_files/train/SerieA_0405.json
Reading file: json_files/train/SerieA_0910.json
Reading file: json_files/train/SerieA_0203.json
Reading file: json_files/train/SerieA_0809.json
Reading file: json_files/train/SerieA_0506.json
Reading file: json_files/train/SerieA_0001.json
Reading file: json_files/train/SerieA_1011.json
Reading file: json_files/eval/SerieA_1617.json
Reading file: json_files/eval/SerieA_1920.json
Reading file: json_files/eval/SerieA_1516.json
Reading file: json_files/eval/SerieA_1819.j

In [2]:
import json
from collections import defaultdict
from datetime import datetime
from enum import IntEnum

def create_team_enum(teams_dict):
    enum_dict = {
        team.upper().replace(" ", "_"): i
        for i, team in enumerate(teams_dict.keys())
    }
    return IntEnum("SerieATeams", enum_dict)

def attach_enum_to_teams(teams_dict, enum_cls):
    return {
        team: (enum_cls[team.upper().replace(" ", "_")], points)
        for team, points in teams_dict.items()
    }



def clean_match(match: dict) -> dict:
    """Remove unnecessary keys from match."""
    keys_to_remove = {"Location", "Group"}
    return {k: v for k, v in match.items() if k not in keys_to_remove}


def update_team_points(match: dict, teams: dict):
    """Update team points based on match result."""
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    teams.setdefault(home, 0)
    teams.setdefault(away, 0)
    
    if home_score > away_score:
        teams[home] += 3
    elif home_score < away_score:
        teams[away] += 3
    else:
        teams[home] += 1
        teams[away] += 1



def track_past_match(match: dict, past_matches: dict, season_number: str):
    """
    Store past match results with matchDay in milliseconds since epoch.
    """
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    key = f"{home}-{away}"
    result = f"{home_score}-{away_score}"
    match_day_str = match["DateUtc"]  # esempio: "2015-08-22 18:00:00Z"

    # Converti la stringa in datetime (senza T)
    match_dt = datetime.strptime(match_day_str, "%Y-%m-%d %H:%M:%SZ")

    # Converti in millisecondi dall'epoca
    match_day_m = int(match_dt.timestamp() * 1000)

    # Inizializza lista se non esiste
    past_matches.setdefault(key, [])
    past_matches[key].append({"result": result, "matchDay": match_day_m, "season": season_number})

# Parse JSON content and update statistics
def parse_json(json_content: str, teams: dict, past_matches: dict, season_number: str):
    """Parse JSON content and update statistics."""
    try:
        matches = json.loads(json_content)

        if not isinstance(matches, list):
            print("Expected a list of matches")
            return []

        cleaned_matches = []

        for match in matches:
            match = clean_match(match)

            if match["HomeTeamScore"] is not None and match["AwayTeamScore"] is not None:
                update_team_points(match, teams)
                track_past_match(match, past_matches, season_number)

            cleaned_matches.append(match)

        return cleaned_matches

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []

# Parse all datasets and update teams and past matches
def parse_dataset(file_contents: dict, teams: dict, past_matches: dict):
    """Parse all files in a dataset."""
    parsed_contents = {}

    for file, content in file_contents.items():    
        season_number = file.split("_")[1].split(".")[0]  # Extract season number from filename
        print(f"Parsing file: {file} (Season {season_number})")
        parsed_contents[file] = parse_json(content, teams, past_matches, season_number)

    return parsed_contents

# ELO rating system is a method for calculating the relative skill levels of players in competitor-versus-competitor games. In this case, we can use it to calculate the skill levels of football teams based on their performance in matches. The ELO rating can be updated after each match based on the result and the expected outcome.
def attach_elo_to_teams(teams_dict):
    """Calculate and attach ELO ratings to teams."""
    base_elo = 1500
    teams_dict_with_elo = {}
    for team, (enum_val, points) in teams_dict.items():
        # Simple ELO calculation based on points (this is a placeholder, you can use a more sophisticated method)
        elo = base_elo
        teams_dict_with_elo[team] = (enum_val, points, elo)
    return teams_dict_with_elo

############################################################################################################################################################

# Structures
all_possible_past_matches_train = defaultdict(list)
all_possible_past_matches_eval = defaultdict(list)
all_possible_past_matches_testing = defaultdict(list)

all_possible_teams_train = {}
all_possible_teams_eval = {}
all_possible_teams_testing = {}

total_matches = 0
# Parse datasets
train_parsed_contents = parse_dataset(
    train_file_contents, all_possible_teams_train, all_possible_past_matches_train
)

eval_parsed_contents = parse_dataset(
    eval_file_contents, all_possible_teams_eval, all_possible_past_matches_eval
)

testing_parsed_contents = parse_dataset(
    testing_file_contents, all_possible_teams_testing, all_possible_past_matches_testing
)

# combine all possible past matches from train, eval and testing
all_possible_past_matches = defaultdict(list, {**all_possible_past_matches_train, **all_possible_past_matches_eval, **all_possible_past_matches_testing})
counter = 0
for match_key, match_list in all_possible_past_matches.items():
    counter += len(match_list)
print(f"Total matches parsed: {counter}")

# combine all possible teams from train, eval and testing
all_possible_teams = {**all_possible_teams_train, **all_possible_teams_eval, **all_possible_teams_testing}

SerieATeams = create_team_enum(all_possible_teams)
num_teams = len(SerieATeams)

all_possible_teams_train = attach_enum_to_teams(all_possible_teams_train, SerieATeams)
all_possible_teams_eval = attach_enum_to_teams(all_possible_teams_eval, SerieATeams)
all_possible_teams_testing = attach_enum_to_teams(all_possible_teams_testing, SerieATeams)

# Attach ELO ratings to teams
all_possible_teams_train = attach_elo_to_teams(all_possible_teams_train)
all_possible_teams_train = {team: (enum_val, points, elo) for team, (enum_val, points, elo) in all_possible_teams_train.items()}

all_possible_teams_eval = attach_elo_to_teams(all_possible_teams_eval)
all_possible_teams_eval = {team: (enum_val, points, elo) for team, (enum_val, points, elo) in all_possible_teams_eval.items()}

all_possible_teams_testing = attach_elo_to_teams(all_possible_teams_testing)
all_possible_teams_testing = {team: (enum_val, points, elo) for team, (enum_val, points, elo) in all_possible_teams_testing.items()}

# sorting all_possible_past_matches by matchDay
for match_key, match_list in all_possible_past_matches_train.items():
    match_list.sort(key=lambda x: x["matchDay"])

# print("\nAll possible past matches sorted by matchDay:")
# for match_key, match_list in all_possible_past_matches_train.items():
#     print(f"{match_key}: {match_list}")

# creating a dict with all possible match of a team, saving: match result and matchDay ordering by matchDay
team_matches_train = defaultdict(list)
team_matches_eval = defaultdict(list)
team_matches_testing = defaultdict(list)

for match_key, match_list in all_possible_past_matches_train.items():
    home_team, away_team = match_key.split("-")
    for match in match_list:
        team_matches_train[home_team].append({"match": match_key, "result": match["result"], "matchDay": match["matchDay"], "season": match["season"]})
        team_matches_train[away_team].append({"match": match_key, "result": match["result"], "matchDay": match["matchDay"], "season": match["season"]})

for match_key, match_list in all_possible_past_matches_eval.items():
    home_team, away_team = match_key.split("-")
    for match in match_list:
        team_matches_eval[home_team].append({"match": match_key, "result": match["result"], "matchDay": match["matchDay"], "season": match["season"]})
        team_matches_eval[away_team].append({"match": match_key, "result": match["result"], "matchDay": match["matchDay"], "season": match["season"]})

for match_key, match_list in all_possible_past_matches_testing.items():
    home_team, away_team = match_key.split("-")
    for match in match_list:
        team_matches_testing[home_team].append({"match": match_key, "result": match["result"], "matchDay": match["matchDay"], "season": match["season"]})
        team_matches_testing[away_team].append({"match": match_key, "result": match["result"], "matchDay": match["matchDay"], "season": match["season"]})

# sorting team_matches by matchDay
for team, matches in team_matches_train.items():
    matches.sort(key=lambda x: x["matchDay"])
for team, matches in team_matches_eval.items():
    matches.sort(key=lambda x: x["matchDay"])
for team, matches in team_matches_testing.items():
    matches.sort(key=lambda x: x["matchDay"])

Parsing file: SerieA_9798.json (Season 9798)
Parsing file: SerieA_9900.json (Season 9900)
Parsing file: SerieA_9596.json (Season 9596)
Parsing file: SerieA_9899.json (Season 9899)
Parsing file: SerieA_9495.json (Season 9495)
Parsing file: SerieA_0607.json (Season 0607)
Parsing file: SerieA_9697.json (Season 9697)
Parsing file: SerieA_0708.json (Season 0708)
Parsing file: SerieA_0102.json (Season 0102)
Parsing file: SerieA_0304.json (Season 0304)
Parsing file: SerieA_0405.json (Season 0405)
Parsing file: SerieA_0910.json (Season 0910)
Parsing file: SerieA_0203.json (Season 0203)
Parsing file: SerieA_0809.json (Season 0809)
Parsing file: SerieA_0506.json (Season 0506)
Parsing file: SerieA_0001.json (Season 0001)
Parsing file: SerieA_1011.json (Season 1011)
Parsing file: SerieA_1617.json (Season 1617)
Parsing file: SerieA_1920.json (Season 1920)
Parsing file: SerieA_1516.json (Season 1516)
Parsing file: SerieA_1819.json (Season 1819)
Parsing file: SerieA_1718.json (Season 1718)
Parsing fi

In [3]:
# Feature engineering and model training would go here, using the parsed data.


# using matchday to find the last 5 matches of a team before a given matchday
def find_previous5_team_matches(team, matchday, team_matches):
    # prendi tutte le partite della squadra
    matches = team_matches.get(team, [])
    
    # filtra solo quelle precedenti al matchday dato
    previous_matches = [m for m in matches if m["matchDay"] < matchday]

    # ritorna le ultime 5 (già ordinate per matchDay)
    return previous_matches[-5:]


def previous5_points(team, previous5_matches):
    points = 0
    for match in previous5_matches:
        home_team, away_team = match["match"].split("-")[0], match["match"].split("-")[1]
        result = match["result"]
        home_score, away_score = int(result.split("-")[0]), int(result.split("-")[1])
        
        if team == home_team:        
            if home_score > away_score:
                points += 3
            elif home_score == away_score:
                points += 1
        elif team == away_team:
            if away_score > home_score:
                points += 3
            elif away_score == home_score:
                points += 1
    return points

def previous5_goals(team, previous5_matches):
    goals = 0
    for match in previous5_matches:
        home_team, away_team = match["match"].split("-")[0], match["match"].split("-")[1]
        result = match["result"]
        home_score, away_score = result.split("-")[0], result.split("-")[1]

        if team == home_team:
            goals += int(home_score)
        elif team == away_team:
            goals += int(away_score)
    return goals

def season_avg_goals_scored(team, matchday,season, team_matches):
    goals = 0
    matches_count = 0
    matches = team_matches.get(team, [])
    # calculate average goals scored by team in this season of all matches previous to the matchday of the current match
    for match in matches:
        if match["season"] == season and match["matchDay"] < matchday:
            home_team, away_team = match["match"].split("-")[0], match["match"].split("-")[1]
            result = match["result"]
            home_score, away_score = result.split("-")[0], result.split("-")[1]

            if team == home_team:
                goals += int(home_score)
                matches_count += 1
            elif team == away_team:
                goals += int(away_score)
                matches_count += 1
    return round(goals / matches_count, 2) if matches_count > 0 else 0

def season_avg_goals_conceded(team, matchday, season, team_matches):
    goals_conceded = 0
    matches_count = 0
    matches = team_matches.get(team, [])
    # calculate average goals conceded by team in this season of all matches previous to the matchday of the current match
    for match in matches:
        if match["season"] == season and match["matchDay"] < matchday:
            home_team, away_team = match["match"].split("-")[0], match["match"].split("-")[1]
            result = match["result"]
            home_score, away_score = result.split("-")[0], result.split("-")[1]

            if team == home_team:
                goals_conceded += int(away_score)
                matches_count += 1
            elif team == away_team:
                goals_conceded += int(home_score)
                matches_count += 1
    return round(goals_conceded / matches_count, 2) if matches_count > 0 else 0

def previous5_goal_diff(team, previous5_matches):
    goal_diff = 0
    for match in previous5_matches:
        home_team, away_team = match["match"].split("-")[0], match["match"].split("-")[1]
        result = match["result"]
        home_score, away_score = result.split("-")[0], result.split("-")[1]

        if team == home_team:
            goal_diff += int(home_score) - int(away_score)
        elif team == away_team:
            goal_diff += int(away_score) - int(home_score)
    return goal_diff

def previous5_points_mean(team, previous5_matches):
    points = previous5_points(team, previous5_matches)
    return round(points / len(previous5_matches), 2) if len(previous5_matches) > 0 else 0

def find_last2yrs_match_history(team1, team2, season):
    # convert season string "1516" to int 1516
    season_int = int(season)
    first = season_int // 100
    second = season_int % 100

    # convert back to string format "1415"
    previous_season = f"{first-1:02d}{second-1:02d}"
    two_seasons_ago = f"{first-2:02d}{second-2:02d}"

    # get all matches between team1 and team2 in the last 2 seasons (previous_season and two_seasons_ago)
    all_possible_past_matches_list = all_possible_past_matches.get(f"{team1}-{team2}", [])
    # given matchday return the two matchday before in the list of all_possible_past_matches_list
    match_history = []
    for match in all_possible_past_matches_list:
        if match["season"] in [previous_season, two_seasons_ago]:
            match_history.append(match)
    return match_history

def create_label(home_goals, away_goals):
    if home_goals > away_goals:
        return 0   # vittoria casa
    elif home_goals == away_goals:
        return 1   # pareggio
    else:
        return 2   # vittoria ospite

def build_features_and_labels(matches, all_possible_teams, team_matches):
    features_teams_ids_list = []
    features_list = []
    labels_list = []
    for match_key, results in matches.items():
        for result in results:
            home_team, away_team = match_key.split("-")[0], match_key.split("-")[1]
            home_team_previous5_matches = find_previous5_team_matches(home_team, result["matchDay"], team_matches)
            away_team_previous5_matches = find_previous5_team_matches(away_team, result["matchDay"], team_matches)
            season_num = result["season"]

            football_match = {
                "home_team_name": all_possible_teams[match_key.split("-")[0]][0],
                "away_team_name": all_possible_teams[match_key.split("-")[1]][0],
                "home_team_goals": int(result["result"].split("-")[0]),
                "away_team_goals": int(result["result"].split("-")[1]),
                "home_previous5_points": previous5_points(match_key.split("-")[0], home_team_previous5_matches),
                "away_previous5_points": previous5_points(match_key.split("-")[1], away_team_previous5_matches),
                "home_previous5_goals": previous5_goals(match_key.split("-")[0], home_team_previous5_matches),
                "away_previous5_goals": previous5_goals(match_key.split("-")[1], away_team_previous5_matches),
                "home_avg_goals_scored": season_avg_goals_scored(match_key.split("-")[0], result["matchDay"], season_num, team_matches),
                "away_avg_goals_scored": season_avg_goals_scored(match_key.split("-")[1], result["matchDay"], season_num, team_matches),
                "home_avg_goals_conceded": season_avg_goals_conceded(match_key.split("-")[0], result["matchDay"], season_num, team_matches),
                "away_avg_goals_conceded": season_avg_goals_conceded(match_key.split("-")[1], result["matchDay"], season_num, team_matches),
                "elo_diff": all_possible_teams[match_key.split("-")[0]][2] - all_possible_teams[match_key.split("-")[1]][2],
                "previous5_goal_diff_home": previous5_goal_diff(match_key.split("-")[0], home_team_previous5_matches),
                "previous5_goal_diff_away": previous5_goal_diff(match_key.split("-")[1], away_team_previous5_matches),
                "previous5_points_mean_home": previous5_points_mean(match_key.split("-")[0], home_team_previous5_matches),
                "previous5_points_mean_away": previous5_points_mean(match_key.split("-")[1], away_team_previous5_matches),
                "last2yrs_match_history": find_last2yrs_match_history(match_key.split("-")[0], match_key.split("-")[1], result["season"]),
                "matchday": result["matchDay"],
                "season": season_num,
                "final_result": result["result"]
            }

            features_teams_ids = [
                football_match["home_team_name"],
                football_match["away_team_name"],
            ]

            features = [
                football_match["home_previous5_points"],
                football_match["away_previous5_points"],
                football_match["home_previous5_goals"],
                football_match["away_previous5_goals"],
                football_match["home_avg_goals_scored"],
                football_match["away_avg_goals_scored"],
                football_match["home_avg_goals_conceded"],
                football_match["away_avg_goals_conceded"],
                football_match["elo_diff"],
                football_match["previous5_goal_diff_home"],
                football_match["previous5_goal_diff_away"],
                # football_match["previous5_points_mean_home"],
                # football_match["previous5_points_mean_away"],
                # football_match["matchday"],
                # football_match["season"],
            ]
            label = create_label(football_match["home_team_goals"], football_match["away_team_goals"])

            features_teams_ids_list.append(features_teams_ids)
            features_list.append(features)
            labels_list.append(label)
    return features_teams_ids_list, features_list, labels_list

############################################################################################################################################################

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

teams_ids_train, features_list_train, Y_train = build_features_and_labels(all_possible_past_matches_train, all_possible_teams_train, team_matches_train)
features_list_train = scaler.fit_transform(features_list_train)

# combine teams_ids_train and features_list_train into X_train
X_train = []
for i in range(len(features_list_train)):
    X_train.append(list(teams_ids_train[i]) + list(features_list_train[i]))
features_num =  len(X_train[0]) if X_train else 0

print("X_train features:", X_train[10])
print("Y_train label:", Y_train[10])
print()
teams_ids_eval, features_list_eval, Y_eval = build_features_and_labels(all_possible_past_matches_eval, all_possible_teams_eval, team_matches_eval)
features_list_eval = scaler.transform(features_list_eval)
# combine teams_ids_eval and features_list_eval into X_eval
X_eval = []
for i in range(len(features_list_eval)):
    X_eval.append(list(teams_ids_eval[i]) + list(features_list_eval[i]))

teams_ids_testing, features_list_testing, Y_testing = build_features_and_labels(all_possible_past_matches_testing, all_possible_teams_testing, team_matches_testing)
features_list_testing = scaler.transform(features_list_testing)
# combine teams_ids_testing and features_list_testing into X_testing
X_testing = []
for i in range(len(features_list_testing)):
    X_testing.append(list(teams_ids_testing[i]) + list(features_list_testing[i]))

X_train features: [<SerieATeams.BARI: 2>, <SerieATeams.PARMA: 3>, np.float64(0.41779045523449654), np.float64(0.663344792590387), np.float64(-0.13297738168800327), np.float64(0.1585218374148139), np.float64(0.18251650593672453), np.float64(-1.7066648241244786), np.float64(0.1535421146886206), np.float64(-1.1225693981450728), np.float64(0.0), np.float64(0.025174802939301775), np.float64(0.9158080810102437)]
Y_train label: 2



In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MatchOutcomePredictor(nn.Module):
    def __init__(self, num_teams, num_features, num_classes=3, embedding_size=10):
        super(MatchOutcomePredictor, self).__init__()

        self.team_embedding = nn.Embedding(num_teams, embedding_size)

        # Le prime 2 colonne sono gli id delle squadre: home_team e away_team
        # Vengono sostituite dai rispettivi embedding
        input_size = (num_features - 2) + (2 * embedding_size)

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.26)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.26)

        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        home_team_embedded = self.team_embedding(x[:, 0].long())
        away_team_embedded = self.team_embedding(x[:, 1].long())

        x = torch.cat((home_team_embedded, away_team_embedded, x[:, 2:]), dim=1)

        x = self.dropout1(torch.relu(self.bn1(self.fc1(x))))
        x = self.dropout2(torch.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        return x

model = MatchOutcomePredictor(num_teams=num_teams, num_features=features_num)
print(model)

# Define a custom Dataset
class MatchDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


batch_size = 32

train_dataset = MatchDataset(X_train, Y_train)
eval_dataset = MatchDataset(X_eval, Y_eval)
test_dataset = MatchDataset(X_testing, Y_testing)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


MatchOutcomePredictor(
  (team_embedding): Embedding(52, 10)
  (fc1): Linear(in_features=31, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.26, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.26, inplace=False)
  (fc3): Linear(in_features=64, out_features=3, bias=True)
)


In [5]:
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Conta quante volte compare ogni classe nel train
class_counts = np.bincount(Y_train, minlength=3)

# Evita divisioni per zero
class_counts = np.where(class_counts == 0, 1, class_counts)

# Peso inversamente proporzionale alla frequenza
class_weights = len(Y_train) / (3 * class_counts)

class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) #  1e-4 or 1e-5
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

Class counts: [2726 1631 1363]
Class weights: tensor([0.6994, 1.1690, 1.3989])


In [6]:
from sklearn.metrics import accuracy_score, classification_report
import copy

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_targets, all_preds)

    return epoch_loss, acc, all_preds, all_targets

num_epochs = 50
best_eval_acc = 0.0
best_state_dict = None

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    eval_loss, eval_acc, _, _ = evaluate(model, eval_loader, criterion, device)
    scheduler.step(eval_loss)
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Eval Loss: {eval_loss:.4f} | "
        f"Eval Accuracy: {eval_acc:.4f} | "
    )

    
    if eval_acc >= best_eval_acc:
        best_eval_acc = eval_acc
        best_state_dict = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state_dict)

test_loss, test_acc, test_preds, test_targets = evaluate(model, test_loader, criterion, device)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

print(classification_report(
    test_targets,
    test_preds,
    target_names=["Home Win", "Draw", "Away Win"]
))

# Spiegazione delle metriche in italiano ed in modo semplice:
print("Spiegazione delle metriche:")
print("\t- Precision: La precision misura la percentuale di predizioni corrette per una classe specifica rispetto a tutte le predizioni fatte per quella classe. Ad esempio, se il modello predice 'Home Win' 100 volte e 80 di queste sono corrette, la precisione per 'Home Win' è dell'80%.")
print("\t- Recall: La recall misura la percentuale di elementi di una classe specifica che sono stati correttamente identificati dal modello rispetto a tutti gli elementi effettivi di quella classe. Ad esempio, se ci sono 100 match che sono 'Home Win' e il modello ne identifica 80, la recall per 'Home Win' è dell'80%.")
print("\t- F1-Score: L'F1-Score è la media armonica della precisione e della recall, fornendo una singola metrica che bilancia entrambe. È utile quando si vuole considerare sia la precisione che la recall in una sola misura.")

Epoch 1/50 | Train Loss: 1.0834 | Eval Loss: 1.0165 | Eval Accuracy: 0.5087 | 
Epoch 2/50 | Train Loss: 1.0517 | Eval Loss: 1.0101 | Eval Accuracy: 0.5171 | 
Epoch 3/50 | Train Loss: 1.0396 | Eval Loss: 1.0208 | Eval Accuracy: 0.5124 | 
Epoch 4/50 | Train Loss: 1.0367 | Eval Loss: 1.0159 | Eval Accuracy: 0.5029 | 
Epoch 5/50 | Train Loss: 1.0325 | Eval Loss: 1.0237 | Eval Accuracy: 0.4866 | 
Epoch 6/50 | Train Loss: 1.0253 | Eval Loss: 1.0254 | Eval Accuracy: 0.4997 | 
Epoch 7/50 | Train Loss: 1.0243 | Eval Loss: 1.0247 | Eval Accuracy: 0.4908 | 
Epoch 8/50 | Train Loss: 1.0187 | Eval Loss: 1.0229 | Eval Accuracy: 0.4892 | 
Epoch 9/50 | Train Loss: 1.0138 | Eval Loss: 1.0185 | Eval Accuracy: 0.4908 | 
Epoch 10/50 | Train Loss: 1.0074 | Eval Loss: 1.0344 | Eval Accuracy: 0.4929 | 
Epoch 11/50 | Train Loss: 1.0047 | Eval Loss: 1.0221 | Eval Accuracy: 0.4860 | 
Epoch 12/50 | Train Loss: 1.0035 | Eval Loss: 1.0325 | Eval Accuracy: 0.4681 | 
Epoch 13/50 | Train Loss: 1.0039 | Eval Loss: 1.0